In [1]:
import torch
from transformers import CLIPImageProcessor, CLIPVisionModel
import sys
sys.path.append("../face_anon_simple")

from diffusers import AutoencoderKL, DDPMScheduler
from diffusers.utils import load_image, make_image_grid
from src.diffusers.models.referencenet.referencenet_unet_2d_condition import (
    ReferenceNetModel,
)
from src.diffusers.models.referencenet.unet_2d_condition import UNet2DConditionModel
from src.diffusers.pipelines.referencenet.pipeline_referencenet_final import (
    StableDiffusionReferenceNetPipeline,
)

In [2]:
custom_cache_dir = "face_anon_simple/new_models"  # Change this to a directory with more space

face_model_id = "hkung/face-anon-simple"
clip_model_id = "openai/clip-vit-large-patch14"
sd_model_id = "stabilityai/stable-diffusion-2-1"

print("Start unet download")
unet = UNet2DConditionModel.from_pretrained(
    face_model_id, subfolder="unet", use_safetensors=True, cache_dir=custom_cache_dir
)
print("Finished unet download")

print("Start Reference net download")
referencenet = ReferenceNetModel.from_pretrained(
    face_model_id, subfolder="referencenet", use_safetensors=True, cache_dir=custom_cache_dir
)
print("Finished reference net download")

print("Start condition reference net download")
conditioning_referencenet = ReferenceNetModel.from_pretrained(
    face_model_id, subfolder="conditioning_referencenet", use_safetensors=True, cache_dir=custom_cache_dir
)
print("Finished condition reference net download")

vae = AutoencoderKL.from_pretrained(
    sd_model_id, subfolder="vae", use_safetensors=True, cache_dir=custom_cache_dir
)
scheduler = DDPMScheduler.from_pretrained(
    sd_model_id, subfolder="scheduler", use_safetensors=True, cache_dir=custom_cache_dir
)
feature_extractor = CLIPImageProcessor.from_pretrained(
    clip_model_id, use_safetensors=True, cache_dir=custom_cache_dir
)
image_encoder = CLIPVisionModel.from_pretrained(
    clip_model_id, use_safetensors=True, cache_dir=custom_cache_dir
)

pipe = StableDiffusionReferenceNetPipeline(
    unet=unet,
    referencenet=referencenet,
    conditioning_referencenet=conditioning_referencenet,
    vae=vae,
    feature_extractor=feature_extractor,
    image_encoder=image_encoder,
    scheduler=scheduler,
)

# pipe = pipe.to("cuda")

generator = torch.manual_seed(1)

Start unet download


/projectnb/cs585bp/projects/face_anonymization_proj/.conda/face-anon-simple/lib/python3.8/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Finished unet download
Start Reference net download
Finished reference net download
Start condition reference net download
Finished condition reference net download


In [3]:
from transformers import AutoImageProcessor, ViTForImageClassification
from PIL import Image
import torch
# Load the model and processor
custom_cache_dir_class="../classifier_gender/classifier_model"
model_name = "rizvandwiki/gender-classification-2"

classifier_model = ViTForImageClassification.from_pretrained(model_name, cache_dir=custom_cache_dir_class)
processor = AutoImageProcessor.from_pretrained(model_name, cache_dir=custom_cache_dir_class)

# 'gender_dataset/CelebA_HQ_face_gender_dataset/train/male'
# image_path ="./my_dataset/train/celeb/real/01758_09704.png"
#[Failed for guy3]
# image = Image.open(image_path).convert("RGB") ""

# Finetuning

In [4]:
import os
import gc
import torch
import time
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.utils import save_image
from PIL import Image
import gc 
from tqdm import tqdm, trange
import face_alignment
from utils.anonymize_faces_in_image import anonymize_faces_in_image
from torch.utils.data import Subset
from torch.utils.data import random_split
import numpy as np
import torch.nn.functional as F
import torchvision.transforms.functional as TF
# !pip install matplotlib
import matplotlib.pyplot as plt
from diffusers.utils import load_image
from collections import Counter
from collections import defaultdict
import random
from sklearn.model_selection import StratifiedShuffleSplit
from torch.cuda.amp import autocast, GradScaler

import bitsandbytes.optim as bnb_optim


import torch.nn as nn
import torchvision.models as models

# Load VGG16 feature extractor





## GPU trial

In [ ]:

# Reduce fragmentation
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:32"
torch.cuda.empty_cache()

# fa = face_alignment.FaceAlignment(face_alignment.LandmarksType.TWO_D, face_detector="sfd")
# Devices
# classifier_device = torch.device("cuda:0")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("CUDA is not available.")

x = torch.randn(1).to("cuda:0")
print(f"Tensor is on: {x.device}") 
classifier_device = torch.device("cuda:0")
pipeline_device = torch.device("cuda:0")
vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_FEATURES).features[:16].eval().to(classifier_device)  # relu3_3
for param in vgg.parameters():
    param.requires_grad = False

# Perceptual loss function
def perceptual_loss(x, y):
    return nn.functional.l1_loss(vgg(x), vgg(y))
# Dataset
transform_raw = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor()
])
dataset = ImageFolder(
    root="PATH OF TRAINING FOLDER OF FACE IMAGES",
    transform=transform_raw
)
print(dataset)
female_indices = range(200)
male_indices = range(21000, 21200)
balanced_indices = list(female_indices) + list(male_indices)

dataset = Subset(dataset, balanced_indices)
train_len = int(0.8 * len(dataset))
val_len = int(0.1 * len(dataset))
test_len = len(dataset) - train_len - val_len
split_generator = torch.Generator().manual_seed(2)

train_set, val_set, test_set = random_split(
    dataset, [train_len, val_len, test_len], generator=split_generator
)

batch_size = 1
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_set, batch_size=1, shuffle=False)
test_loader = DataLoader(test_set, batch_size=1, shuffle=False)

def get_class_distribution(loader):
    class_counts = Counter()
    for images, labels in loader:
        class_counts[labels.item()] += 1
    return dict(class_counts)

print("Train class distribution:", get_class_distribution(train_loader))
print("Validation class distribution:", get_class_distribution(val_loader))
print("Test class distribution:", get_class_distribution(test_loader))

# Classifier
classifier_model.to(classifier_device).eval()
for param in classifier_model.parameters():
    param.requires_grad = False

# Pipeline setup
pipe.to(pipeline_device)
pipe.unet.train()
pipe.referencenet.eval()
pipe.conditioning_referencenet.eval()
pipe.vae.eval()
for param in pipe.conditioning_referencenet.parameters(): 
    param.requires_grad = False
for param in pipe.vae.parameters(): 
    param.requires_grad = False
for param in pipe.referencenet.parameters(): 
    param.requires_grad = False



# Optimizer and loss
optimizer = optim.Adam(list(pipe.unet.parameters()), lr=5e-6)
criterion = nn.CrossEntropyLoss()
# reconstruction_loss_fn = nn.MSELoss()

val_change=0

for param in pipe.unet.parameters(): 
    # print("Classifier frozen")
    param.requires_grad = True
face_encoder = models.resnet18(weights="IMAGENET1K_V1")
face_encoder.fc = torch.nn.Identity()  # remove final classifier
face_encoder = face_encoder.to(classifier_device).eval()
for param in face_encoder.parameters():
    param.requires_grad = False


# for name, param in pipe.unet.named_parameters():
#     if any(target in name for target in ["up_blocks.3", "conv_out"]):  # <-- modify this
#         print(f"✅ Training: {name}")
#         param.requires_grad = True
#     else:
#         # print(f"🧊 Frozen: {name}")
#         param.requires_grad = False


# Classifier input transform
transform_for_classifier = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

# Training loop
num_epochs = 15
generator = torch.manual_seed(1)
torch.manual_seed(1)
np.random.seed(1)

train_losses = []
val_losses = []
val_losses_recon=[]


for epoch in trange(num_epochs, desc="Training Epochs"):
    print(f"\n🌀 Epoch {epoch+1}/{num_epochs}")
    epoch_train_loss = 0.0
    epoch_train_loss_recon = 0.0
    epoch_val_loss_recon=0.0
    total_train_batches = 0
    epoch_val_loss = 0.0
    total_val_batches = 0


    for i, (image_tensor, label) in enumerate(train_loader):
        try:
            gc.collect()
            torch.cuda.empty_cache()

            ce_loss = 0
            perceptual_losses = []
            embedding_losses = []

            for j in range(image_tensor.shape[0]):
                original_image = transforms.ToPILImage()(image_tensor[j]).convert("RGB")
                seed = int(time.time())  # or: random.randint(0, 2**32 - 1)
                # Set seeds
                generator = torch.manual_seed(seed)
                torch.manual_seed(seed)
                np.random.seed(seed)
                generated_image_tensor = pipe(
                    source_image=original_image,
                    conditioning_image=original_image,
                    num_inference_steps=9,
                    guidance_scale=4.0,
                    generator=generator,
                    output_type='pt',
                    anonymization_degree=1.25,
                ).images[0]

                processed_tensor = transform_for_classifier(generated_image_tensor)
                inputs = {"pixel_values": processed_tensor.unsqueeze(0)}
                logits = classifier_model(**inputs).logits
                target = label[j].unsqueeze(0).to(classifier_device)

                ce_loss += criterion(logits, target)

                # Compute perceptual loss
                resized_original = transforms.Resize(generated_image_tensor.shape[-2:])(image_tensor[j]).to(classifier_device)
                perceptual_losses.append(
                    perceptual_loss(
                        generated_image_tensor.to(classifier_device).unsqueeze(0), 
                        resized_original.unsqueeze(0)
                    )
                )
                # For PIL Images (original_image)
                preprocess_pil = transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Resize((224, 224)),
                    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
                ])

                # For already-Tensors (generated_image_tensor)
                preprocess_tensor = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),
                ])

                # Now this works fine
                preprocessed_generated_face = preprocess_tensor(generated_image_tensor).unsqueeze(0).to(classifier_device)
                preprocessed_original_face = preprocess_pil(original_image).unsqueeze(0).to(classifier_device)

                face_embed_generated = face_encoder(preprocessed_generated_face)
                face_embed_original = face_encoder(preprocessed_original_face)

                cos = torch.nn.CosineSimilarity(dim=1, eps=1e-6)
                similarity = cos(face_embed_generated, face_embed_original)
                embedding_loss = (1 - similarity).mean()
                embedding_losses.append(embedding_loss)

            reconstruction_loss = torch.stack(perceptual_losses).mean()
            embedding_loss_total = torch.stack(embedding_losses).mean()

            print(f"Reconstruction (Perceptual) Loss: {reconstruction_loss.item():.4f}")
            print(f"Cross-Entropy Loss: {ce_loss.item():.4f}")
            print(f"Embedding Loss: {embedding_loss_total.item():.4f}")


            # if ce_loss.item() < 0.5:
            #     reconstruction_loss = torch.tensor(0.0, device=classifier_device)

            loss = ce_loss + 0.1 * reconstruction_loss + embedding_loss_total

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()
            epoch_train_loss_recon+=reconstruction_loss
            total_train_batches += 1     

            if i % 1 == 0:
                print(f"  Batch {i:03d} | Loss: {loss.item():.4f}")
            gc.collect()
            torch.cuda.empty_cache()
            classifier_model.eval()
            pipe.unet.eval()
            if i % 40 == 0:
                val_change+=1
                with torch.no_grad():
                    for val_idx, (val_image_tensor, val_label) in enumerate(val_loader):
                        val_image = transforms.ToPILImage()(val_image_tensor[0]).convert("RGB")
                        torch.manual_seed(1)
                        np.random.seed(1)
                        generator = torch.manual_seed(1)
                        generated_val_tensor = pipe(
                            source_image=val_image,
                            conditioning_image=val_image,
                            num_inference_steps=9,
                            guidance_scale=4.0,
                            generator=generator,
                            output_type='pt',
                            anonymization_degree=1.25,
                        ).images[0]

                        processed_val_tensor = transform_for_classifier(generated_val_tensor)
                        original_resized = transforms.Resize(generated_val_tensor.shape[-2:])(
                            val_image_tensor[0].squeeze(0)
                        ).to(classifier_device)

                        val_inputs = {"pixel_values": processed_val_tensor.unsqueeze(0)}
                        val_logits = classifier_model(**val_inputs).logits
                        val_target = val_label[0].unsqueeze(0).to(classifier_device)

                        resized_val_original = transforms.Resize(generated_val_tensor.shape[-2:])(
                            val_image_tensor[j]
                        ).to(classifier_device)

                        reconstruction_loss = perceptual_loss(
                            generated_val_tensor.to(classifier_device).unsqueeze(0),
                            resized_val_original.unsqueeze(0)
                        )

                        val_loss = criterion(val_logits, val_target)
                        epoch_val_loss += val_loss.item()
                        epoch_val_loss_recon+=reconstruction_loss
                        total_val_batches += 1

                        # 💾 Save first validation image
                        
                        # print("Saving validation example")
                        if val_idx in range(10):
                            out_dir = f"outputs_embeddings_new/gender_finetune/epoch_{epoch}/validation/image{val_idx}"
                            os.makedirs(out_dir, exist_ok=True)
                            save_path = os.path.join(out_dir, f"val_step_{val_change}_image_label_{val_label[0].item()}.png")
                            # print(save_path)
                            save_path_og = os.path.join(out_dir, f"val_image_label_{val_label[0].item()}_original.png")
                            save_image(generated_val_tensor.detach().cpu().clamp(0, 1), save_path)
                            save_image(val_image_tensor[0].cpu(), save_path_og)

                avg_val_loss = epoch_val_loss / total_val_batches if total_val_batches > 0 else 0
                avg_val_loss_recon = epoch_val_loss_recon/total_val_batches if total_val_batches > 0 else 0
                val_losses.append(avg_val_loss)
                val_losses_recon.append(avg_val_loss_recon)
                print(f"🔎 Validation Loss: {avg_val_loss:.4f}")

                # Reset pipeline mode back to train for referencenet
                pipe.unet.train()
            
        except RuntimeError as e:
            if "out of memory" in str(e):
                print(e)
                print(f"⚠️ OOM at batch {i}, skipping")
                torch.cuda.empty_cache()
                # raise e 
                continue
            else:
                print("ERRORRRRRR : ",e)
                # continue
                raise e
    
    save_dir = f"checkpoints_embeddings/epoch_{epoch}"
    os.makedirs(save_dir, exist_ok=True)
    torch.save(pipe.unet.state_dict(), os.path.join(save_dir, "unet.pt"))
    print(f"💾 Saved UNet at {save_dir}")
    train_losses=epoch_train_loss/total_train_batches
    train_recon_loss=epoch_train_loss_recon/total_train_batches
    np.save(f"{save_dir}/train_losses_epoch{epoch}.npy", np.array(train_losses))
    np.save(f"{save_dir}/train_losses_recon_epoch{epoch}.npy", np.array(train_losses))
    np.save(f"{save_dir}/val_losses_epoch{epoch}.npy", np.array(val_losses))
    np.save(f"{save_dir}/val_losses_recon_epoch{epoch}.npy", np.array([loss.cpu().numpy() for loss in val_losses_recon]))


GPU 0: NVIDIA A100 80GB PCIe
Tensor is on: cuda:0
Dataset ImageFolder
    Number of datapoints: 23999
    Root location: /projectnb/cs585bp/projects/face_anonymization_proj/face_anon_simple/gender_dataset/CelebA_HQ_face_gender_dataset/train
    StandardTransform
Transform: Compose(
               Resize(size=(512, 512), interpolation=bilinear, max_size=None, antialias=warn)
               ToTensor()
           )
Train class distribution: {0: 153, 1: 167}
Validation class distribution: {0: 23, 1: 17}
Test class distribution: {0: 24, 1: 16}


Training Epochs:   0%|          | 0/15 [00:00<?, ?it/s]


🌀 Epoch 1/15


/projectnb/cs585bp/projects/face_anonymization_proj/.conda/face-anon-simple/lib/python3.8/site-packages/torchvision/transforms/functional.py:1603: UserWarning: The default value of the antialias parameter of all the resizing transforms (Resize(), RandomResizedCrop(), etc.) will change from None to True in v0.17, in order to be consistent across the PIL and Tensor backends. To suppress this warning, directly pass antialias=True (recommended, future default), antialias=None (current default, which means False for Tensors and True for PIL), or antialias=False (only works on Tensors - PIL will still use antialiasing). This also applies if you are using the inference transforms from the models weights: update the call to weights.transforms(antialias=True).
  warnings.warn(


Reconstruction (Perceptual) Loss: 0.9459
Cross-Entropy Loss: 0.0038
Embedding Loss: 0.0271
  Batch 000 | Loss: 0.1254
🔎 Validation Loss: 0.7361
Reconstruction (Perceptual) Loss: 0.7635
Cross-Entropy Loss: 0.0705
Embedding Loss: 0.0265
  Batch 001 | Loss: 0.1734
Reconstruction (Perceptual) Loss: 0.6726
Cross-Entropy Loss: 0.0088
Embedding Loss: 0.0238
  Batch 002 | Loss: 0.0999
Reconstruction (Perceptual) Loss: 0.7348
Cross-Entropy Loss: 0.0052
Embedding Loss: 0.0209
  Batch 003 | Loss: 0.0996
Reconstruction (Perceptual) Loss: 0.7170
Cross-Entropy Loss: 0.0077
Embedding Loss: 0.0167
  Batch 004 | Loss: 0.0961
Reconstruction (Perceptual) Loss: 0.9279
Cross-Entropy Loss: 0.0035
Embedding Loss: 0.0376
  Batch 005 | Loss: 0.1338
Reconstruction (Perceptual) Loss: 0.8975
Cross-Entropy Loss: 1.9419
Embedding Loss: 0.0442
  Batch 006 | Loss: 2.0758
Reconstruction (Perceptual) Loss: 0.8444
Cross-Entropy Loss: 3.9502
Embedding Loss: 0.0341
  Batch 007 | Loss: 4.0687
Reconstruction (Perceptual) Lo

...


🌀 Epoch 4/15
Reconstruction (Perceptual) Loss: 0.6323
Cross-Entropy Loss: 0.0036
Embedding Loss: 0.0157
  Batch 000 | Loss: 0.0825
🔎 Validation Loss: 0.3136
Reconstruction (Perceptual) Loss: 0.7580
Cross-Entropy Loss: 0.0082
Embedding Loss: 0.0134
  Batch 001 | Loss: 0.0974
Reconstruction (Perceptual) Loss: 0.6395
Cross-Entropy Loss: 0.0037
Embedding Loss: 0.0102
  Batch 002 | Loss: 0.0778
Reconstruction (Perceptual) Loss: 0.7733
Cross-Entropy Loss: 0.0035
Embedding Loss: 0.0114
  Batch 003 | Loss: 0.0922
Reconstruction (Perceptual) Loss: 0.7323
Cross-Entropy Loss: 0.0035
Embedding Loss: 0.0148
  Batch 004 | Loss: 0.0916
Reconstruction (Perceptual) Loss: 0.5050
Cross-Entropy Loss: 0.0066
Embedding Loss: 0.0063
  Batch 005 | Loss: 0.0634
Reconstruction (Perceptual) Loss: 0.7375
Cross-Entropy Loss: 0.0063
Embedding Loss: 0.0284
  Batch 006 | Loss: 0.1085
Reconstruction (Perceptual) Loss: 0.5777
Cross-Entropy Loss: 0.0064
Embedding Loss: 0.0086
  Batch 007 | Loss: 0.0728
Reconstruction (

Training Epochs:  27%|██▋       | 4/15 [3:12:31<8:49:28, 2888.04s/it]


🌀 Epoch 5/15
Reconstruction (Perceptual) Loss: 0.7027
Cross-Entropy Loss: 0.0039
Embedding Loss: 0.0285
  Batch 000 | Loss: 0.1026
🔎 Validation Loss: 0.0767
Reconstruction (Perceptual) Loss: 0.4166
Cross-Entropy Loss: 0.0034
Embedding Loss: 0.0296
  Batch 001 | Loss: 0.0746
Reconstruction (Perceptual) Loss: 0.4029
Cross-Entropy Loss: 0.0060
Embedding Loss: 0.0093
  Batch 002 | Loss: 0.0557
Reconstruction (Perceptual) Loss: 0.7130
Cross-Entropy Loss: 0.0048
Embedding Loss: 0.0225
  Batch 003 | Loss: 0.0987
Reconstruction (Perceptual) Loss: 0.5796
Cross-Entropy Loss: 0.0063
Embedding Loss: 0.0095
  Batch 004 | Loss: 0.0738
Reconstruction (Perceptual) Loss: 0.6795
Cross-Entropy Loss: 0.0037
Embedding Loss: 0.0204
  Batch 005 | Loss: 0.0921
Reconstruction (Perceptual) Loss: 0.8559
Cross-Entropy Loss: 0.0036
Embedding Loss: 0.0299
  Batch 006 | Loss: 0.1190
Reconstruction (Perceptual) Loss: 0.5894
Cross-Entropy Loss: 0.0052
Embedding Loss: 0.0207
  Batch 007 | Loss: 0.0848
Reconstruction (

Training Epochs:  33%|███▎      | 5/15 [4:00:43<8:01:32, 2889.24s/it]


🌀 Epoch 6/15
Reconstruction (Perceptual) Loss: 0.6264
Cross-Entropy Loss: 0.0084
Embedding Loss: 0.0190
  Batch 000 | Loss: 0.0901
🔎 Validation Loss: 0.1127
Reconstruction (Perceptual) Loss: 0.5551
Cross-Entropy Loss: 0.0067
Embedding Loss: 0.0286
  Batch 001 | Loss: 0.0908
Reconstruction (Perceptual) Loss: 0.6778
Cross-Entropy Loss: 0.0046
Embedding Loss: 0.0131
  Batch 002 | Loss: 0.0855
Reconstruction (Perceptual) Loss: 0.8283
Cross-Entropy Loss: 0.0040
Embedding Loss: 0.0223
  Batch 003 | Loss: 0.1091
Reconstruction (Perceptual) Loss: 0.5596
Cross-Entropy Loss: 0.0053
Embedding Loss: 0.0104
  Batch 004 | Loss: 0.0716
Reconstruction (Perceptual) Loss: 0.6070
Cross-Entropy Loss: 0.0065
Embedding Loss: 0.0241
  Batch 005 | Loss: 0.0913
Reconstruction (Perceptual) Loss: 0.7345
Cross-Entropy Loss: 0.0035
Embedding Loss: 0.0203
  Batch 006 | Loss: 0.0973
Reconstruction (Perceptual) Loss: 0.6529
Cross-Entropy Loss: 0.0058
Embedding Loss: 0.0193
  Batch 007 | Loss: 0.0904
Reconstruction (